<h3>실습 안내</h3><p>제공된 합성 PDF <code>E01.pdf</code>를 직접 읽어 쪽별 Document와 청크가 만들어지는 과정을 확인합니다.</p>

In [1]:
# PDF 텍스트 추출과 Document 분할에 필요한 패키지를 설치합니다.
# 이 절은 외부 API를 호출하지 않으므로 .env를 읽지 않습니다.
%pip install -q "pypdf==6.7.5" "langchain==1.4.0" "langchain-text-splitters==1.1.2"

Note: you may need to restart the kernel to use updated packages.


<h3>1. PDF 경로 찾기</h3><p>현재 폴더와 부모 폴더에서 실습 말뭉치 폴더를 찾습니다.</p>

In [2]:
# project_root는 labs/data/rag-corpus가 존재하는 첫 상위 폴더입니다.
# break는 파일을 찾은 즉시 for 반복을 종료합니다.
from pathlib import Path

start = Path.cwd().resolve()
project_root = None
folders = [start]
folders.extend(start.parents)
for folder in folders:
    if (folder / "labs/data/rag-corpus").exists():
        project_root = folder
        break
assert project_root is not None, "실습 말뭉치를 찾지 못했습니다."
pdf_path = project_root / "labs/data/rag-corpus/pdf/easy/E01.pdf"
assert pdf_path.exists(), f"PDF가 없습니다: {pdf_path}"
print("실습 파일:", pdf_path.relative_to(project_root))

실습 파일: labs/data/rag-corpus/pdf/easy/E01.pdf


<h3>2. PDF 한 쪽을 Document로 바꾸기</h3><p>정기 휴관일이 있는 2쪽을 고르고 본문·문서 ID·쪽·파일명을 함께 저장합니다.</p>

In [3]:
# reader.pages[1]의 인덱스 1은 PDF의 두 번째 쪽을 선택합니다.
# Document는 page_content에 본문, metadata에 출처 정보를 보관합니다.
from pypdf import PdfReader
from langchain_core.documents import Document

reader = PdfReader(pdf_path)
page_text = reader.pages[1].extract_text()
document = Document(
    page_content=page_text,
    metadata={"doc_id": "E01", "page": 2, "source": pdf_path.name},
)
print("Document 타입:", type(document).__name__)
print("metadata:", document.metadata)
print("content:", document.page_content[:240])

Document 타입:

 Document
metadata: {'doc_id': 'E01', 'page': 2, 'source': 'E01.pdf'}
content: 1. 도서관 소개
미르시립도서관은 1998년 10월 미르시 중앙동에 문을 연 공공도서관입니다. 지상 4층, 지하 1층 규모
이며 장서 42만 권과 전자책 6만 종을 보유하고 있습니다. 2024년 리모델링을 마치고 3층에 '메이커 스
페이스'와 '조용한 열람실'을 새로 열었습니다.
도서관의 사명은 "누구나, 언제나, 무료로"입니다. 시민 모두가 자료를 자유롭게 이용하고 배울 수 있도
록 돕는 것을 최우선으로 합니다.
2. 이용 


<h3>3. 기본 분할 실행</h3><p>청크 크기 500, 겹침 50으로 나누고 반환된 개수·길이·메타데이터를 보여 줍니다.</p>

In [4]:
# split_documents()는 Document 리스트를 받으므로 [document]로 감쌍니다.
# 반환된 chunks도 Document 리스트이며 원본 metadata가 복사됩니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents([document])
print("청크 수:", len(chunks))
for number, chunk in enumerate(chunks, start=1):
    print(number, "길이:", len(chunk.page_content), "metadata:", chunk.metadata)
    print(chunk.page_content[:100])

청크 수: 2
1 길이: 497 metadata: {'doc_id': 'E01', 'page': 2, 'source': 'E01.pdf'}
1. 도서관 소개
미르시립도서관은 1998년 10월 미르시 중앙동에 문을 연 공공도서관입니다. 지상 4층, 지하 1층 규모
이며 장서 42만 권과 전자책 6만 종을 보유하고 있습니
2 길이: 452 metadata: {'doc_id': 'E01', 'page': 2, 'source': 'E01.pdf'}
구역 평일 주말 비고
종합자료실(2층) 09:00~22:00 09:00~18:00
어린이자료실(1층) 09:00~18:00 09:00~18:00 보호자 동반 권장
디지털자료실(2층


<h3>4. 값 하나만 바꾸어 비교</h3><p><code>chunk_overlap=50</code>은 그대로 두고 <code>chunk_size</code>만 500에서 300으로 줄입니다.</p>

In [5]:
# smaller와 splitter의 차이는 chunk_size 하나뿐입니다.
# 기본·변경 결과의 개수와 첫 청크 길이를 같은 형식으로 비교합니다.
smaller = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
small_chunks = smaller.split_documents([document])
print("기본 500:", len(chunks), "개, 첫 길이", len(chunks[0].page_content))
print("변경 300:", len(small_chunks), "개, 첫 길이", len(small_chunks[0].page_content))
assert len(small_chunks) >= len(chunks)

기본 500: 2 개, 첫 길이 497
변경 300: 4 개, 첫 길이 242


<h3>정리</h3><p>Document의 metadata는 분할 후에도 남았고, 청크 크기만 줄이자 청크 개수와 길이가 바뀌었습니다.</p>

<p><strong>원인</strong> — 겹침을 청크 크기보다 크게 지정하면 스플리터를 만들 수 없습니다.</p>

In [6]:
# chunk_overlap=501은 chunk_size=500보다 크므로 ValueError가 발생합니다.
# except는 예상한 ValueError만 포착해 Run All이 계속되게 합니다.
try:
    RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=501)
except ValueError as error:
    print("예상한 실패:", error)

예상한 실패: Got a larger chunk overlap (501) than chunk size (500), should be smaller.


<p><strong>수정</strong> — 겹침을 50으로 줄이고 같은 Document를 정상적으로 나눕니다.</p>

In [7]:
# fixed_splitter는 청크 크기보다 작은 겹침을 사용합니다.
# fixed_chunks에 청크가 있고 모두 E01 메타데이터를 가지는지 확인합니다.
fixed_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
fixed_chunks = fixed_splitter.split_documents([document])
print("After 청크 수:", len(fixed_chunks))
assert fixed_chunks
for chunk in fixed_chunks:
    assert chunk.metadata["doc_id"] == "E01"

After 청크 수: 2
